In [1]:
print('/knkng')

/knkng


In [2]:
%pwd
import os
os.chdir("../")
#Just went one step bcakword with this directory....

%pwd

'd:\\mlops\\Crypto_Guardian'

In [26]:
import sqlite3

conn = sqlite3.connect("artifacts\data_ingestion\sample_data.db")

cursor = conn.cursor()

texts = cursor.execute(
    ''' SELECT text,label FROM comment'''
).fetchall()
conn.commit()


texts

Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "C:\Users\sahil\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py", line 3579, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\sahil\AppData\Local\Temp\ipykernel_29104\2975192334.py", line 7, in <module>
    texts = cursor.execute(
sqlite3.OperationalError: no such table: comment

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\sahil\AppData\Roaming\Python\Python310\site-packages\IPython\core\interactiveshell.py", line 2170, in showtraceback
    stb = self.InteractiveTB.structured_traceback(
  File "C:\Users\sahil\AppData\Roaming\Python\Python310\site-packages\IPython\core\ultratb.py", line 1457, in structured_traceback
    return FormattedTB.structured_traceback(
  File "C:\Users\sahil\AppData\Roaming\Python\Python310\site-packages\IPython\core\ultratb.py", line 1348, in structured_traceback
    return Verbo

In [4]:
len(texts)

2061

In [29]:
def start_db():
    conn = sqlite3.connect("sample_data.db")
    cursor = conn.cursor()
    
    # cursor.execute(''' ALTER TABLE comments ADD COLUMN label TEXT''')
    cursor.execute(''' ALTER TABLE comments ADD COLUMN score float32''')
    
    conn.commit()
    

In [14]:
texts[:5]

[('He was super Kangaroo Jacked!',),
 ('Arkansas don&#39;t play man',),
 ('kangaroo listens to NBA young boy',),
 ('Who’s watching in 2001',),
 ('Ray making a new show on Youtube made me recap some =3 videos like this one. Won&#39;t forget this fucking kangoroo.',)]

In [8]:
from src.Crypto import logger
import feedparser
import pandas as pd
from transformers import pipeline
import torch


C:\Users\sahil\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [34]:
# start_db()

device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")
sentiment_pipeline = pipeline("sentiment-analysis", device=device)
conn = sqlite3.connect("sample_data.db")
cursor = conn.cursor()
for i in range(len(texts[:5])):
    # print(f"texts: {texts[i][0]}")
    results = sentiment_pipeline(texts[i][0])
    # print("Results: \n " ,results[0])
    # print("Label: " ,results[0]['label'])
    # print("Score: " ,results[0]['score'])
    
    cursor.execute(""" UPDATE comments set "label" = ? ,'score'=?  where "text" = ?""",(results[0]['label'],results[0]['score'],texts[i][0]))
    
    conn.commit()

    
    
    


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Using device: CPU
[04-05-2026 03:39:53 PM - _client - INFO - HTTP Request: HEAD https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/resolve/714eb0f/config.json "HTTP/1.1 307 Temporary Redirect"]
[04-05-2026 03:39:53 PM - _client - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/714eb0fa89d2f80546fda750413ed43d93601a13/config.json "HTTP/1.1 200 OK"]


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5524.20it/s]


[04-05-2026 03:39:54 PM - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/714eb0f/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"]
[04-05-2026 03:39:54 PM - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"]
texts: He was super Kangaroo Jacked!
Label:  NEGATIVE
Score:  0.9905241131782532
texts: Arkansas don&#39;t play man
Label:  NEGATIVE
Score:  0.976599395275116
texts: kangaroo listens to NBA young boy
Label:  POSITIVE
Score:  0.9686216115951538
texts: Who’s watching in 2001
Label:  POSITIVE
Score:  0.9969611763954163
texts: Ray making a new show on Youtube made me recap some =3 videos like this one. Won&#39;t forget this fucking kangoroo.
Label:  NEGATIVE
Score:  0.9811485409736633


In [40]:
texts[10:15]

[('This kangaroo remainds me of ricardo milos😅😂😂',),
 ('super video ray excellent..i m new youtuber.',),
 ('Why do I live in Arkansas',),
 ('June 3 2019 ayyy🔥🔥🔥',),
 ('&quot;Do you even fuck bro? &quot;',)]

In [17]:
results[0]['score']


0.9811485409736633

# All Done All Research Has Been Done Now Lets Make the Pipeline

In [9]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataLabellingConfig:
    database_path : Path
    
from src.Crypto.constants import *
from src.Crypto.utils.helper import read_yaml,create_directories
from src.Crypto import logger

class ConfigurationManager:
    def __init__(self,config_file_path=CONFIG_FILE_PATH,
                      params_file_path = PARAMS_FILE_PATH,
                      schema_file_path = SCHEMA_FILE_PATH):
        self.config =read_yaml( config_file_path)
        self.parmas = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)
        
        # create_directories([self.config.artifacts_root])
        logger.debug(f"Till Now all the Yaml Files are Read Sucessfully...✅")
   
    def get_data_labelling(self)->DataLabellingConfig:
        config = self.config.data_labelling
        # create_directories([config.database_path])
        
        data_labelling_config = DataLabellingConfig(
            database_path=config.database_path
            
        )
        logger.debug("get_data_labelling is working compeletely fine...✅")
        return data_labelling_config
from src.Crypto import logger
import feedparser
import pandas as pd
from transformers import pipeline
import torch
from sources import Links,links
import sqlite3
from googleapiclient.discovery import build
from dotenv import load_dotenv
import sqlite3

class DataLabellingComponent:
    def __init__(self,config:DataLabellingConfig):
        self.config = config
        
        self.conn = sqlite3.connect(self.config.database_path)
        self.cursor = self.conn.cursor()

    
    def fetch_database(self):
        text = self.cursor.execute(
                ''' SELECT text FROM comments'''
            ).fetchall()
        self.conn.commit()
        
        return text
    
    # def alter_db(self):
    #     self.cursor.execute(''' ALTER TABLE comments ADD COLUMN label TEXT''')
    #     self.cursor.execute(''' ALTER TABLE comments ADD COLUMN score float32''')
    #     self.conn.commit()
        
    def label_db(self):
        
        device = 0 if torch.cuda.is_available() else -1
        print(f"Using device: {'GPU' if device == 0 else 'CPU'}")
        sentiment_pipeline = pipeline("sentiment-analysis", device=device)
        # conn = sqlite3.connect("sample_data.db")
        # cursor = conn.cursor()
        
        texts = self.fetch_database()
        print( "texts: \n ",texts[10:20])
        for i in range(len(texts[10:20])):
            # print(f"texts: {texts[i][0]}")
            results = sentiment_pipeline(texts[i][0])
            # print("Results: \n " ,results[0])
            # print("Label: " ,results[0]['label'])
            # print("Score: " ,results[0]['score'])
            
            self.cursor.execute(""" UPDATE comments set "label" = ? ,'score'=?  where "text" = ?""",(results[0]['label'],results[0]['score'],texts[i][0]))
            self.conn.commit()
        print("All Done...✅")
                    

try:
    cfm = ConfigurationManager()
    di = cfm.get_data_labelling()
    dic = DataLabellingComponent(di)
    dic.fetch_database()
    # dic.alter_db()
    dic.label_db()
    
except Exception as e:
    raise e

[04-05-2026 07:02:51 PM - helper - INFO - Yaml File :config\config.yaml Read Sucessfully✅]
[04-05-2026 07:02:51 PM - helper - INFO - Yaml File :params.yaml Read Sucessfully✅]
[04-05-2026 07:02:51 PM - helper - INFO - Yaml File :schema.yaml Read Sucessfully✅]


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Using device: CPU
[04-05-2026 07:02:52 PM - _client - INFO - HTTP Request: HEAD https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/resolve/714eb0f/config.json "HTTP/1.1 307 Temporary Redirect"]
[04-05-2026 07:02:52 PM - _client - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/714eb0fa89d2f80546fda750413ed43d93601a13/config.json "HTTP/1.1 200 OK"]


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2451.55it/s]


[04-05-2026 07:02:52 PM - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/714eb0f/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"]
[04-05-2026 07:02:53 PM - _client - INFO - HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"]
texts: 
  [('This kangaroo remainds me of ricardo milos😅😂😂',), ('super video ray excellent..i m new youtuber.',), ('Why do I live in Arkansas',), ('June 3 2019 ayyy🔥🔥🔥',), ('&quot;Do you even fuck bro? &quot;',), ('Goddamn he is funny. I lost it when he said this might be his cell mate. 😂😂',), ('Dude the last vid was insanity',), ('2019?!?',), ('It’s been 4 years. This was the last real Ray episode of Equals 3.',), ('Y&#39;all watching in 2018?',)]
All Done...✅
